In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
cd assistant-rh

In [ ]:
# ================================================================
# CSV -> Q/A (grouped) -> Embeddings (bge-m3) -> Postgres rag_chunks_2
# ================================================================
from __future__ import annotations
import os, re, json, hashlib
from pathlib import Path
from typing import List, Iterable, Dict

import pandas as pd
import numpy as np
import psycopg
from psycopg.rows import dict_row
from sentence_transformers import SentenceTransformer


DB_USER = os.getenv("PGUSER", "")
DB_PASS = os.getenv("PGPASSWORD", "")
DB_NAME = os.getenv("PGDATABASE", "")

DB_HOST = os.getenv("PGHOST", "127.0.0.1")
DB_PORT = 10001          # the port shown by `db-tunnel`
DB_SSL  = os.getenv("PGSSLMODE", "disable")

import torch

# -------------------------
# Config
# -------------------------
CSV_PATH     = os.getenv("RGRH_CSV_PATH", "./data/rgrh/entretien_professionnelle/all_RGRH_entretien_professionnelle_rules_population_statut.csv")
SCHEMA       = os.getenv("RGRH_SCHEMA", "public")
TABLE        = os.getenv("RGRH_TABLE", "rag_chunks_3")
FQTN         = f'{SCHEMA}.{TABLE}'
EMBED_DB_COL = os.getenv("EMBEDDING_COLUMN_DB", "embedding_m3")
MODEL_NAME   = "BAAI/bge-m3"        # FR-friendly
BATCH_SIZE   = 64
NORMALIZE    = True

SOURCE_NAME  = Path(CSV_PATH).name
OUT_JSONL    = os.getenv("RGRH_OUT_JSONL", "./data/out/chunked/chunks_from_csv_grouped_entretien_professionnelle.jsonl")  # optionnel: pour audit

# -------------------------
# Columns in your CSV
# -------------------------
COL_DOMAINE     = "Libellé du Domaine"
COL_SSDOMAINE   = "Libellé du Sous-domaine"
COL_EVENT       = "Libellé de l'évènement"
COL_POP_CODE    = "Codification Population"
COL_POP_LABEL   = "Libellé Population"
COL_RULE_CODE   = "Code de la règle"
COL_RULE_LABEL  = "Règle littérale"
COL_PASSANT     = "Passant / Exclu"          # optionnel
COL_GP          = "Général / Particulier"    # optionnel
COL_DEBUT       = "Date de début"            # optionnel
COL_FIN         = "Date de fin"              # optionnel
COL_REF         = "Références juridiques"   

# -------------------------
# Helpers
# -------------------------
def sha1_u(s: str) -> str:
    return hashlib.sha1((s or "").encode("utf-8")).hexdigest()

def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def find_statut_cols(cols: Iterable[str]) -> List[str]:
    # détecte dynamiquement les colonnes “statut”
    out = []
    for c in cols:
        cl = c.lower()
        if "statut" in cl or "statu" in cl:  # capture variantes/accents
            out.append(c)
    return out

def pg_conn():
    return psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASS,
        sslmode="disable",
        row_factory=dict_row,
        connect_timeout=10,
    )

def ensure_vector_column(conn, schema: str, table: str, col: str, dim: int):
    GET_VECTOR_DIM_SQL = """
    SELECT (att.atttypmod - 4) AS dim
    FROM pg_attribute att
    JOIN pg_class cls   ON cls.oid = att.attrelid
    JOIN pg_namespace n ON n.oid = cls.relnamespace
    JOIN pg_type t      ON t.oid  = att.atttypid
    WHERE n.nspname = %s AND cls.relname = %s AND att.attname = %s
      AND att.attnum > 0 AND NOT att.attisdropped AND t.typname = 'vector';
    """
    GET_COL_INFO_SQL = """
    SELECT data_type, udt_name
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s AND column_name = %s;
    """
    with conn.cursor() as cur:
        cur.execute(GET_VECTOR_DIM_SQL, (schema, table, col))
        row = cur.fetchone()
        if row and row["dim"] is not None:
            db_dim = int(row["dim"])
            if db_dim == dim:
                print(f"✓ {schema}.{table}.{col} already vector({dim})")
                return
            print(f"→ ALTER COLUMN {schema}.{table}.{col} vector({db_dim}) -> vector({dim})")
            cur.execute(f'ALTER TABLE "{schema}"."{table}" ALTER COLUMN "{col}" TYPE vector({dim});')
            return

        cur.execute(GET_COL_INFO_SQL, (schema, table, col))
        info = cur.fetchone()
        if info is None:
            print(f"→ ADD COLUMN {schema}.{table}.{col} vector({dim})")
            cur.execute(f'ALTER TABLE "{schema}"."{table}" ADD COLUMN "{col}" vector({dim});')
            return

        raise RuntimeError(f"Column {schema}.{table}.{col} exists but is type ({info['data_type']}/{info['udt_name']}), not vector.")

def vec_literal(v) -> str:
    if isinstance(v, np.ndarray): v = v.tolist()
    return "[" + ",".join(f"{float(x):.7f}" for x in v) + "]"

# -------------------------
# 1) Read CSV
# -------------------------
df = pd.read_csv(CSV_PATH, dtype=str).fillna("")
df["source"]="RGRH"
needed = [COL_DOMAINE, COL_SSDOMAINE, COL_EVENT, COL_POP_CODE, COL_POP_LABEL, COL_RULE_CODE,COL_REF]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Colonnes manquantes dans le CSV : {missing}")

# tidy
for col in set(needed + [COL_RULE_LABEL, COL_PASSANT, COL_GP, COL_DEBUT, COL_FIN,COL_REF]):
    if col in df.columns:
        df[col] = df[col].astype(str).map(norm)

STATUT_COLS = find_statut_cols(df.columns)

# -------------------------
# 2) Group -> build Q/A
# -------------------------
group_keys = [COL_DOMAINE, COL_SSDOMAINE, COL_EVENT, COL_POP_CODE, COL_POP_LABEL] + STATUT_COLS
groups = df.groupby(group_keys, dropna=False, sort=False)

records: List[Dict] = []

for key, g in groups:
    keyd = dict(zip(group_keys, key))
    domaine   = keyd.get(COL_DOMAINE, "")
    ssdomaine = keyd.get(COL_SSDOMAINE, "")
    event     = keyd.get(COL_EVENT, "")
    pop_code  = keyd.get(COL_POP_CODE, "")
    pop_label = keyd.get(COL_POP_LABEL, "")

    # contexte optionnel
    passant_vals = sorted({norm(x) for x in g[COL_PASSANT].unique()}) if COL_PASSANT in g else []
    gp_vals      = sorted({norm(x) for x in g[COL_GP].unique()}) if COL_GP in g else []

    # section/thematique
    section_path = " > ".join([p for p in [domaine, ssdomaine, event] if p])
    thematique   = "/".join([p for p in [domaine, ssdomaine] if p])
    # références juridiques (grouped, unique, joined with ';')
    refs = [norm(x) for x in g.get(COL_REF, pd.Series([], dtype=str)).unique() if norm(x)]
    # preserve order and deduplicate while keeping first occurrence
    seen_r = set()
    refs_unique = []
    for r in refs:
        if r not in seen_r:
            seen_r.add(r)
            refs_unique.append(r)
    refs_joined = ";".join(refs_unique)

    # statuts (si colonnes détectées)
    statut_lines = []
    for sc in STATUT_COLS:
        vals = sorted({norm(x) for x in g[sc].unique() if norm(x)})
        if vals:
            statut_lines.append(f"{sc} : {', '.join(vals)}")

    # règles (littérales)
    rules = []
    for _, row in g.iterrows():
        code = norm(row.get(COL_RULE_CODE, ""))
        if not code:
            continue
        lib  = norm(row.get(COL_RULE_LABEL, ""))
        rules.append(f"{code} — {lib}" if lib else code)

    seen = set()
    rules_unique = []
    for r in rules:
        if r not in seen:
            seen.add(r)
            rules_unique.append(r)

    # Question
    headline = " > ".join([p for p in [domaine, ssdomaine, event] if p])
    pop_part = f"{pop_code}" + (f" ({pop_label})" if pop_label else "")
    q_ctx = []
    if gp_vals:      q_ctx.append(f"Cadre : {', '.join([v for v in gp_vals if v])}")
    if passant_vals: q_ctx.append(f"Applicabilité : {', '.join([v for v in passant_vals if v])}")
    if statut_lines: q_ctx.extend(statut_lines)

    question = f"{headline} — Quelles règles s’appliquent à la population {pop_part} ?"
    if q_ctx:
        question += "\n" + "\n".join(q_ctx)

    # Réponse littérale (liste des règles)
    if rules_unique:
        answer = "Règles applicables :\n" + "\n".join(f"• {r}" for r in rules_unique)
    else:
        answer = "Le document ne mentionne aucune règle applicable pour cette combinaison."

    # IDs stables
    qa_key = f"d:{domaine}|sd:{ssdomaine}|e:{event}|pop:{pop_code}|pl:{pop_label}"
    for sc in STATUT_COLS:
        qa_key += f"|{sc}:{keyd.get(sc,'')}"
    qa_id = sha1_u(qa_key)

    # Q_ONLY
    text_q = question
    hash_q = sha1_u(f"{SOURCE_NAME}|{qa_id}|Q_ONLY|0|{text_q[:256]}")
    records.append({
        "hash_id": hash_q,
        "qa_id": qa_id,
        "parent_qa_id": None,
        "source_name": SOURCE_NAME,
        "section_path": section_path,
        "role": "Q_ONLY",
        "chunk_index": 0,
        "text": text_q,
        "lang": "fr",
        "thematique": thematique,
        "references_juridiques": refs_joined,
    })

    # QA_COMPOSITE
    composite = f"Q: {question}\n\nR: {answer}"
    hash_c = sha1_u(f"{SOURCE_NAME}|{qa_id}|QA_COMPOSITE|1|{composite[:256]}")
    records.append({
        "hash_id": hash_c,
        "qa_id": qa_id,
        "parent_qa_id": None,
        "source_name": SOURCE_NAME,
        "section_path": section_path,
        "role": "QA_COMPOSITE",
        "chunk_index": 1,
        "text": composite,
        "lang": "fr",
        "thematique": thematique,
        "references_juridiques": refs_joined,  
    })

chunks = pd.DataFrame(records)
# add source so inserted rows carry their origin
chunks['source'] = 'RGRH'
print(f"Built chunks: {len(chunks)} rows from {len(groups)} groups.")



In [ ]:
chunks.head(5)

In [ ]:
# (optionnel) sauver pour audit
Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in chunks.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"JSONL écrit: {OUT_JSONL}")

# -------------------------
# 3) Embeddings (BAAI/bge-m3)
# -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer(MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {device}")

def format_passage(txt: str) -> str:
    # bge-m3: pas de préfixe
    return txt or ""

corpus = [format_passage(t) for t in chunks["text"].astype(str).tolist()]
vecs_all = []
for i in range(0, len(corpus), BATCH_SIZE):
    batch = corpus[i:i+BATCH_SIZE]
    vecs = model.encode(batch,
                        batch_size=len(batch),
                        normalize_embeddings=NORMALIZE,
                        convert_to_numpy=True,
                        show_progress_bar=True)
    vecs_all.append(vecs)
emb = np.vstack(vecs_all).astype(np.float32)   # (N, 1024)

chunks["embedding_str"] = ["[" + ",".join(f"{float(x):.7f}" for x in v) + "]" for v in emb]

# -------------------------
# 4) Ensure DB column & insert
# -------------------------
#with pg_conn() as con:
 #   ensure_vector_column(con, SCHEMA, TABLE, EMBED_DB_COL, emb.shape[1])
  #  con.commit()

SQL = f"""
INSERT INTO "{SCHEMA}"."{TABLE}"
(hash_id, qa_id, parent_qa_id, source_name, section_path, role, chunk_index, text, "{EMBED_DB_COL}", lang, thematique, source , references_juridiques)
VALUES (%(hash_id)s, %(qa_id)s, %(parent_qa_id)s, %(source_name)s, %(section_path)s, %(role)s, %(chunk_index)s, %(text)s, %(embedding_str)s::vector, %(lang)s, %(thematique)s, %(source)s, %(references_juridiques)s)
ON CONFLICT (hash_id) DO UPDATE SET
  qa_id = EXCLUDED.qa_id,
  parent_qa_id = EXCLUDED.parent_qa_id,
  source_name = EXCLUDED.source_name,
  section_path = EXCLUDED.section_path,
  role = EXCLUDED.role,
  chunk_index = EXCLUDED.chunk_index,
  text = EXCLUDED.text,
  "{EMBED_DB_COL}" = EXCLUDED.{EMBED_DB_COL},
  lang = EXCLUDED.lang,
  thematique = EXCLUDED.thematique,
  source = EXCLUDED.source,
  references_juridiques = EXCLUDED.references_juridiques;
"""

BATCH_DB = 2000
with pg_conn() as con, con.cursor() as cur:
    for i in range(0, len(chunks), BATCH_DB):
        cur.executemany(SQL, chunks.iloc[i:i+BATCH_DB].to_dict(orient="records"))
        con.commit()

print(f"Inserted candidates: {len(chunks)} (conflicts skipped).")
